In [1]:
# ===============================
# Kaggle: Color×Illumination Benchmark (no OpenCV)
# Attach your dataset via "Add data" (right panel). This cell auto-detects it.
# ===============================
%pip -q install timm==0.9.16 einops==0.7.0 scikit-image==0.24.0 scikit-learn==1.6.1 pandas==2.2.2 matplotlib==3.8.4

import os, random, math, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision as tv
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from skimage import color as skcolor

# ---------------- Config ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Expected taxonomy
colors = ["Black","Blue","Gray","Orange","Pink","Purple","Skyblue","White","Yellow"]
conds  = ["fluorescentLight","indoor","indoorNight","sunLight"]
img_exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

# ---------------- Auto-detect dataset under /kaggle/input ----------------
def looks_like_dataset(path: Path) -> bool:
    if not path.exists(): return False
    found_colors = [c for c in colors if (path / c).exists()]
    if len(found_colors) < 3:
        return False
    for c in found_colors:
        if any((path / c / k).exists() for k in conds):
            return True
    return False

INPUT_ROOT = Path("/kaggle/input")
assert INPUT_ROOT.exists(), "Kaggle input root missing. Did you click 'Add data'?"
candidates = []
scan = [INPUT_ROOT] + [p for p in INPUT_ROOT.rglob("*") if p.is_dir() and len(p.relative_to(INPUT_ROOT).parts) <= 4]
for base in scan:
    try:
        if looks_like_dataset(base):
            candidates.append(base)
    except Exception:
        pass
if not candidates:
    raise RuntimeError("Could not find <Color>/<Illumination> folders under /kaggle/input. Ensure dataset is attached and unzipped.")
DATASET_ROOT = sorted(candidates, key=lambda p: len(str(p)), reverse=True)[0]
print("✅ Using DATASET_ROOT:", DATASET_ROOT)

# ---------------- Build manifest ----------------
rows = []
for c in colors:
    for k in conds:
        d = DATASET_ROOT / c / k
        if not d.exists():
            print("WARNING: missing", d)
            continue
        for f in d.iterdir():
            if f.is_file() and f.suffix.lower() in img_exts:
                rows.append({"rel_path": str(f.relative_to(DATASET_ROOT)),
                             "color": c, "illumination": k})

df = pd.DataFrame(rows)
assert len(df)>0, "No images found; expected <Color>/<Illumination>/*"
df["illumination"] = df["illumination"].replace({"sunlight":"sunLight","Sunlight":"sunLight","sun Light":"sunLight"})
df["bucket"] = df["color"] + "|" + df["illumination"]
print("Total images:", len(df))

OUTDIR = Path("/kaggle/working"); OUTDIR.mkdir(exist_ok=True, parents=True)
(df.assign(root=str(DATASET_ROOT))).to_csv(OUTDIR/"manifest.csv", index=False)

# ---------------- Stratified 70/15/15 over joint label ----------------
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(sss1.split(df, df["bucket"]))
trainval = df.iloc[trainval_idx].reset_index(drop=True)
test     = df.iloc[test_idx].reset_index(drop=True)

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)  # ≈15% of total
train_idx, val_idx = next(sss2.split(trainval, trainval["bucket"]))
train = trainval.iloc[train_idx].reset_index(drop=True)
val   = trainval.iloc[val_idx].reset_index(drop=True)

print("Split sizes:", len(train), len(val), len(test))
train.to_csv(OUTDIR/"train.csv", index=False)
val.to_csv(OUTDIR/"val.csv", index=False)
test.to_csv(OUTDIR/"test.csv", index=False)

# ---------------- Torch datasets & loaders ----------------
class ColorDataset(Dataset):
    def __init__(self, df, root, transform=None):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform
        self.c2i = {c:i for i,c in enumerate(colors)}
        self.k2i = {k:i for i,k in enumerate(conds)}
        self.yc  = self.df["color"].map(self.c2i).values.astype(np.int64)
        self.yk  = self.df["illumination"].map(self.k2i).values.astype(np.int64)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root / row["rel_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(self.yc[idx]), int(self.yk[idx]), row["rel_path"]

img_size = 224
train_tf = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # conservative; avoid hue jitter
    transforms.ToTensor(),
])
test_tf  = transforms.Compose([transforms.Resize((img_size,img_size)), transforms.ToTensor()])

train_dl = DataLoader(ColorDataset(train, DATASET_ROOT, train_tf), batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(ColorDataset(val,   DATASET_ROOT, test_tf),  batch_size=64, shuffle=False, num_workers=2)
test_dl  = DataLoader(ColorDataset(test,  DATASET_ROOT, test_tf),  batch_size=64, shuffle=False, num_workers=2)

# ---------------- Metrics ----------------
def macro_f1(y_true, y_pred): return f1_score(y_true, y_pred, average="macro")
def macro_acc(y_true, y_pred): return accuracy_score(y_true, y_pred)
def ece(probs, labels, n_bins=15):
    confid = probs.max(1); preds = probs.argmax(1); correct = (preds==labels).astype(float)
    bins = np.linspace(0,1,n_bins+1); out = 0.0
    for i in range(n_bins):
        m = (confid>bins[i]) & (confid<=bins[i+1])
        if m.any(): out += m.mean() * abs(correct[m].mean() - confid[m].mean())
    return float(out)

def per_illum_confusions(paths, y_true, y_pred, df_meta):
    res = {}
    meta = pd.DataFrame({"rel_path": paths, "y_true": y_true, "y_pred": y_pred})
    merged = meta.merge(df_meta[["rel_path","illumination"]], on="rel_path", how="left")
    for k in conds:
        sub = merged[merged.illumination == k]
        if len(sub)==0: continue
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=list(range(len(colors))))
        res[k] = cm
    return res

# ---------------- Models & helpers ----------------
def make_model(name, num_classes=9):
    n = name.lower()
    if n == "resnet18":
        m = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m
    if n == "mobilenetv3":
        m = tv.models.mobilenet_v3_large(weights=tv.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m
    if n == "vit_tiny":
        return timm.create_model("vit_tiny_patch16_224.augreg_in21k", pretrained=True, num_classes=num_classes)
    raise ValueError("Unknown model")

# bucket→color class weights (mitigate imbalance)
_t = train.copy()
bc = _t["bucket"].value_counts().to_dict()
bidx = {b:i for i,b in enumerate(sorted(bc))}
bw = 1.0/np.array([bc[b] for b in sorted(bc)], dtype=np.float32); bw = bw/bw.mean()
bucket_w_per_sample = _t["bucket"].map(lambda b:bw[bidx[b]]).values
class_w = pd.DataFrame({"color":_t["color"], "w":bucket_w_per_sample}).groupby("color")["w"].mean().reindex(colors).fillna(1.0).values
class_w_t = torch.tensor(class_w, dtype=torch.float32).to(DEVICE)

class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.t = nn.Parameter(torch.ones(1))
    def forward(self, logits): return logits / self.t.clamp(min=1e-3)

@torch.no_grad()
def eval_loader(model_or_fn, dl):
    if hasattr(model_or_fn, "eval"):
        model_or_fn.eval()
        forward = model_or_fn
    else:
        forward = model_or_fn
    ys, ps, paths = [], [], []
    for x, y, _, p in dl:
        x = x.to(DEVICE)
        logits = forward(x)
        prob = F.softmax(logits, dim=1).cpu().numpy()
        ys.append(y.numpy()); ps.append(prob); paths += list(p)
    y_true = np.concatenate(ys); probs = np.concatenate(ps, axis=0); y_pred = probs.argmax(1)
    return y_true, y_pred, probs, paths

def fit_tempscaling(model, dl):
    model.eval(); logits_list, labels_list = [], []
    with torch.no_grad():
        for x, y, _, _ in dl:
            x = x.to(DEVICE); logits_list.append(model(x).cpu()); labels_list.append(y)
    logits = torch.cat(logits_list, 0); labels = torch.cat(labels_list, 0)
    scaler = TempScaler().to(logits.device)
    opt = torch.optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50); nll = nn.CrossEntropyLoss()
    def closure(): opt.zero_grad(); loss = nll(scaler(logits), labels); loss.backward(); return loss
    opt.step(closure); return scaler

def train_model(name, epochs=6, lr=3e-4, wd=1e-4):
    model = make_model(name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.CrossEntropyLoss(weight=class_w_t)
    best = (-1, 1e9); best_state = None
    for ep in range(1, epochs+1):
        model.train()
        for x, y, _, _ in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = crit(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
        yv, pv, _, _ = eval_loader(model, val_dl)
        mf1 = macro_f1(yv, pv); mac = macro_acc(yv, pv)
        scaler = fit_tempscaling(model, val_dl)
        scaler = scaler.to(DEVICE)   # <<< ensure same device as model
        yv2, pv2, pr2, _ = eval_loader(lambda z: scaler(model(z)), val_dl)
        ecev = ece(pr2, yv2)
        print(f"[{name}] Epoch {ep:02d}  val macroF1={mf1:.4f}  acc={mac:.4f}  ECE={ecev:.4f}")
        if mf1 > best[0]:
            best = (mf1, ecev); best_state = {"m": model.state_dict(), "s": scaler.state_dict()}
    model.load_state_dict(best_state["m"]); scaler = TempScaler(); scaler.load_state_dict(best_state["s"]); scaler = scaler.to(DEVICE)
    return model, scaler

# ---------------- Train/evaluate deep models ----------------
results_main = []
for model_name in ["resnet18","mobilenetv3","vit_tiny"]:
    model, scaler = train_model(model_name, epochs=6, lr=3e-4, wd=1e-4)
    yt, yp, pr, paths = eval_loader(model, test_dl)
    mf1, mac, e_raw = macro_f1(yt, yp), macro_acc(yt, yp), ece(pr, yt)
    logits_list = []
    with torch.no_grad():
        for x,_,_,_ in test_dl:
            x = x.to(DEVICE); logits_list.append(scaler(model(x)).cpu())
    probs_cal = F.softmax(torch.cat(logits_list,0), dim=1).numpy()
    e_cal = ece(probs_cal, yt)
    confs = per_illum_confusions(paths, yt, yp, test)
    results_main.append({"model": model_name, "macroF1": mf1, "macroAcc": mac, "ECE_raw": e_raw, "ECE_cal": e_cal, "confs": confs})

res_df = pd.DataFrame([{k:v for k,v in r.items() if k!='confs'} for r in results_main]).sort_values("macroF1", ascending=False)
print("\n=== In-Distribution Results ===")
print(res_df)
res_df.to_csv(OUTDIR/"results_in_distribution.csv", index=False)

# ---------------- Cross-illumination Shift A/B ----------------
def subset_by_conds(_df, keep): return _df[_df["illumination"].isin(keep)].reset_index(drop=True)
train_A, val_A, test_A = subset_by_conds(train, ["indoor","fluorescentLight"]), subset_by_conds(val, ["indoor","fluorescentLight"]), subset_by_conds(test, ["sunLight","indoorNight"])
train_B, val_B, test_B = subset_by_conds(train, ["sunLight","indoorNight"]), subset_by_conds(val, ["sunLight","indoorNight"]), subset_by_conds(test, ["indoor","fluorescentLight"])

def make_loader_from_df(df_split, tf): return DataLoader(ColorDataset(df_split, DATASET_ROOT, tf), batch_size=64, shuffle=True, num_workers=2)
def run_shift(model_name, train_df, val_df, test_df):
    tr, va = make_loader_from_df(train_df, train_tf), make_loader_from_df(val_df, test_tf)
    te = DataLoader(ColorDataset(test_df, DATASET_ROOT, test_tf), batch_size=64, shuffle=False, num_workers=2)
    _t = train_df.copy(); _t["bucket"] = _t["color"]+"|"+_t["illumination"]
    bc = _t["bucket"].value_counts().to_dict(); bidx = {b:i for i,b in enumerate(sorted(bc))}
    bw = 1.0/np.array([bc[b] for b in sorted(bc)], dtype=np.float32); bw = bw/bw.mean()
    cw = pd.DataFrame({"color":_t["color"],"w":_t["bucket"].map(lambda b:bw[bidx[b]])}).groupby("color")["w"].mean().reindex(colors).fillna(1.0).values
    cw = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
    model = make_model(model_name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=cw)
    for _ in range(6):
        model.train()
        for x,y,_,_ in tr:
            x,y = x.to(DEVICE), y.to(DEVICE)
            loss = crit(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
    scaler = fit_tempscaling(model, va)
    scaler = scaler.to(DEVICE)   # <<< ensure same device as model
    yt, yp, pr, _ = eval_loader(model, te)
    probs_cal = []
    with torch.no_grad():
        for x,_,_,_ in te:
            x = x.to(DEVICE); probs_cal.append(F.softmax(scaler(model(x)), dim=1).cpu().numpy())
    probs_cal = np.concatenate(probs_cal, 0)
    return {"macroF1": macro_f1(yt, yp), "ECE": ece(probs_cal, yt)}

shift_results = []
for m in ["resnet18","mobilenetv3","vit_tiny"]:
    a = run_shift(m, train_A, val_A, test_A)
    b = run_shift(m, train_B, val_B, test_B)
    shift_results.append({"model": m, "macroF1_A": a["macroF1"], "ECE_A": a["ECE"],
                          "macroF1_B": b["macroF1"], "ECE_B": b["ECE"]})
shift_df = pd.DataFrame(shift_results)
print("\n=== Cross-Illumination (Shift A/B) ===")
print(shift_df)
shift_df.to_csv(OUTDIR/"results_shift_ab.csv", index=False)

# ---------------- Classical baselines (Lab histogram via scikit-image) ----------------
def lab_histogram_skimage(img_path, bins=(32,32,32)):
    img = Image.open(img_path).convert("RGB").resize((256,256))
    arr = np.asarray(img).astype(np.float32) / 255.0
    lab = skcolor.rgb2lab(arr)
    L = lab[...,0] * (255.0/100.0); a = (lab[...,1] + 128.0); b = (lab[...,2] + 128.0)
    hist, _ = np.histogramdd(np.stack([L,a,b], axis=-1).reshape(-1,3),
                              bins=bins, range=((0,255),(0,255),(0,255)))
    hist = hist.astype(np.float32); hist = hist / (hist.sum() + 1e-8)
    return hist.flatten()

def make_lab_features(df_split, pca_fit=True, pca=None):
    feats, labels = [], []
    for _, r in df_split.iterrows():
        feats.append(lab_histogram_skimage(DATASET_ROOT / r["rel_path"]))
        labels.append(colors.index(r["color"]))
    X = np.stack(feats, 0); y = np.array(labels)
    if pca_fit:
        pca = PCA(n_components=128, random_state=SEED).fit(X)
    return (pca.transform(X) if pca else X), y, pca

Xtr, ytr, pca = make_lab_features(train, pca_fit=True)
Xte, yte, _   = make_lab_features(test,  pca_fit=False, pca=pca)

svm = SVC(kernel="rbf", probability=True, C=8.0, gamma="scale", random_state=SEED).fit(Xtr, ytr)
rf  = RandomForestClassifier(n_estimators=400, max_depth=None, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)

def eval_classical(clf, X, y):
    prob = clf.predict_proba(X); pred = prob.argmax(1)
    return macro_f1(y, pred), accuracy_score(y, pred), ece(prob, y)

svm_f1, svm_acc, svm_ece = eval_classical(svm, Xte, yte)
rf_f1,  rf_acc,  rf_ece  = eval_classical(rf,  Xte, yte)

classical_df = pd.DataFrame([
    {"model":"SVM-RBF","macroF1":svm_f1,"macroAcc":svm_acc,"ECE":svm_ece},
    {"model":"RandomForest","macroF1":rf_f1,"macroAcc":rf_acc,"ECE":rf_ece}
])
print("\n=== Classical Baselines (Test) ===")
print(classical_df)
classical_df.to_csv(OUTDIR/"results_classical.csv", index=False)

# ---------------- Wrap up ----------------
with open(OUTDIR/"README.txt","w") as f:
    f.write(
        "Artifacts:\n"
        "- manifest.csv (root + rel_path + labels)\n"
        "- train.csv / val.csv / test.csv (stratified 70/15/15 on color×illumination)\n"
        "- results_in_distribution.csv (deep models: macroF1/macroAcc/ECE raw+cal)\n"
        "- results_shift_ab.csv (cross-illumination robustness)\n"
        "- results_classical.csv (SVM/RF on Lab histograms)\n"
    )

print("\nArtifacts saved to /kaggle/working:")
for p in OUTDIR.iterdir(): print(" -", p.name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 27.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 95.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda
✅ Using DATASET_ROOT: /kaggle/input/different-colors-in-challenging-lightening-v2/colorDataset_ml224
Total images: 12085
Split sizes: 8458 1814 1813


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s] 


[resnet18] Epoch 01  val macroF1=0.9963  acc=0.9972  ECE=0.0023
[resnet18] Epoch 02  val macroF1=0.9967  acc=0.9972  ECE=0.0026
[resnet18] Epoch 03  val macroF1=0.9993  acc=0.9994  ECE=0.0019
[resnet18] Epoch 04  val macroF1=0.9948  acc=0.9950  ECE=0.0035
[resnet18] Epoch 05  val macroF1=0.9938  acc=0.9945  ECE=0.0024
[resnet18] Epoch 06  val macroF1=1.0000  acc=1.0000  ECE=0.0000


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
100%|██████████| 21.1M/21.1M [00:00<00:00, 137MB/s] 


[mobilenetv3] Epoch 01  val macroF1=0.7425  acc=0.7696  ECE=0.0527
[mobilenetv3] Epoch 02  val macroF1=0.9718  acc=0.9757  ECE=0.0167
[mobilenetv3] Epoch 03  val macroF1=0.9993  acc=0.9994  ECE=0.0006
[mobilenetv3] Epoch 04  val macroF1=1.0000  acc=1.0000  ECE=0.0000
[mobilenetv3] Epoch 05  val macroF1=0.9984  acc=0.9983  ECE=0.0014
[mobilenetv3] Epoch 06  val macroF1=0.9731  acc=0.9807  ECE=0.0101


model.safetensors:   0%|          | 0.00/39.0M [00:00<?, ?B/s]

[vit_tiny] Epoch 01  val macroF1=0.9866  acc=0.9901  ECE=0.0070
[vit_tiny] Epoch 02  val macroF1=0.9955  acc=0.9967  ECE=0.0038
[vit_tiny] Epoch 03  val macroF1=0.9904  acc=0.9928  ECE=0.0026
[vit_tiny] Epoch 04  val macroF1=0.9670  acc=0.9768  ECE=0.0107
[vit_tiny] Epoch 05  val macroF1=0.9824  acc=0.9868  ECE=0.0054
[vit_tiny] Epoch 06  val macroF1=0.9720  acc=0.9802  ECE=0.0080

=== In-Distribution Results ===
         model   macroF1  macroAcc   ECE_raw   ECE_cal
0     resnet18  1.000000  1.000000  0.001324  0.000002
1  mobilenetv3  0.973003  0.980695  0.010782  0.010796
2     vit_tiny  0.965172  0.975731  0.017506  0.019952

=== Cross-Illumination (Shift A/B) ===
         model  macroF1_A     ECE_A  macroF1_B     ECE_B
0     resnet18   0.798312  0.207695   0.700060  0.231870
1  mobilenetv3   0.869435  0.053159   0.778003  0.175007
2     vit_tiny   0.666377  0.337245   0.815721  0.136321

=== Classical Baselines (Test) ===
          model   macroF1  macroAcc       ECE
0       SVM-R